# Consumer engagement

Fig. 1(d). Match user-video interactions on platform-labeled AIGC to unlabeled content. The duration caliper is 3 seconds and is applied after the distance.

### 1. The two arms

Focal is the treated arm. Comparison is the other arm. Nothing is paired yet.

In [1]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

from core.pipeline import ensure_synthetic, load_config

cfg = load_config(root)
spec = cfg["tasks"]["consumer"]
pool = ensure_synthetic(root, cfg, rebuild=False)[{"creator": "creators", "consumer": "consumers", "video": "videos", "algorithm": "algorithm"}["consumer"]]
focal = pool[pool["treated"] == 1].reset_index(drop=True)
comparison = pool[pool["treated"] == 0].reset_index(drop=True)
import pandas as pd
pd.DataFrame([{
    "focal_label": spec["focal"],
    "comparison_label": spec["comparison"],
    "n_focal": len(focal),
    "n_comparison": len(comparison),
    "exact": ", ".join(spec["exact"]),
    "caliper": ", ".join(f"{name} <= {limit}" for name, limit in (spec.get("caliper") or {}).items()) or "none",
}])


,focal_label,comparison_label,n_focal,n_comparison,exact,caliper
0,platform-labeled AIGC,unlabeled content,2400,2400,"gender, city, category, time_of_day, upload_week",duration_seconds <= 3.0


### 2. Exact cells

Units are blocked on the categorical covariates first. A focal unit whose cell has no comparison unit cannot be matched, before any distance is computed.

In [2]:
from core.matching import exact_cell_counts

cells = exact_cell_counts(focal, comparison, spec["exact"])
cells.sort_values(["n_comparison", "n_focal"], ascending=[True, False])


,gender,city,category,time_of_day,upload_week,n_focal,n_comparison
62,female,south,food,morning,week_1,12,2
175,male,north,leisure,morning,week_4,5,2
135,female,east,beauty,evening,week_4,13,3
234,male,south,beauty,night,week_1,12,3
160,male,north,leisure,noon,week_2,10,3
11,female,north,food,night,week_4,8,3
151,male,north,food,evening,week_4,8,3
219,male,south,leisure,night,week_4,6,3
179,male,north,beauty,noon,week_4,5,3
33,female,north,beauty,noon,week_3,17,4


### 3. Scale of the continuous covariates

Nearest neighbor uses a pooled z-score. The mean and standard deviation are taken over both arms together.

In [3]:
from core.matching import covariate_scale

covariate_scale(focal, comparison, spec["numeric"])


,covariate,pooled_mean,pooled_sd
0,age,31.990270,7.799631
1,duration_seconds,18.114971,5.978199
2,plays_labeled_14d,10.126177,5.417745
3,plays_labeled_60d,28.145904,14.949883
4,plays_unlabeled_14d,33.976070,18.471131
5,plays_unlabeled_60d,93.824617,51.121309
6,watch_time_labeled_14d,22.833114,12.011098
7,watch_time_labeled_60d,61.166759,32.913057
8,watch_time_unlabeled_14d,62.153965,34.182051
9,watch_time_unlabeled_60d,167.357081,87.441237


### 4. Nearest neighbor inside the cell

Each focal unit takes the comparison unit in the same cell with the smallest Euclidean distance. A caliper, when there is one, sets the distance to infinity and that neighbor is dropped. The ledger counts, per cell, how many focal units were kept and how many were dropped.

In [4]:
from core.matching import match_process

pairs, ledger = match_process(
    focal,
    comparison,
    spec["exact"],
    spec["numeric"],
    spec["outcomes"],
    spec.get("caliper") or {},
)
ledger.sort_values(["n_dropped", "n_focal"], ascending=[False, False])


,gender,city,category,time_of_day,upload_week,n_focal,n_comparison,n_matched,n_dropped,median_distance
105,female,east,food,night,week_3,11,6,5,6,4.386333
160,male,north,leisure,noon,week_2,10,3,4,6,3.946283
11,female,north,food,night,week_4,8,3,2,6,4.744453
62,female,south,food,morning,week_1,12,2,7,5,4.653968
213,male,south,leisure,evening,week_3,12,7,7,5,4.673762
10,female,north,food,night,week_1,11,7,6,5,5.133353
66,female,south,leisure,noon,week_1,11,7,6,5,5.017106
4,female,north,food,evening,week_2,9,5,4,5,4.805458
33,female,north,beauty,noon,week_3,17,4,13,4,5.823283
153,male,north,food,night,week_3,13,5,9,4,5.258459


### 5. What was dropped, and how often a comparison unit is reused

Replacement means one comparison unit may be the neighbor for more than one focal unit.

In [5]:
import pandas as pd

reuse = pairs["comparison_row"].value_counts()
pd.DataFrame([{
    "focal_units": len(focal),
    "comparison_units": len(comparison),
    "exact_cells": len(ledger),
    "dropped_no_comparison_in_cell": int(ledger.loc[ledger["n_comparison"] == 0, "n_focal"].sum()),
    "dropped_by_caliper_or_empty_neighborhood": int(ledger.loc[ledger["n_comparison"] > 0, "n_dropped"].sum()),
    "pairs": len(pairs),
    "comparison_units_used": int(reuse.shape[0]),
    "max_reuse": int(reuse.max()) if len(reuse) else 0,
}])


,focal_units,comparison_units,exact_cells,dropped_no_comparison_in_cell,dropped_by_caliper_or_empty_neighborhood,pairs,comparison_units_used,max_reuse
0,2400,2400,288,0,372,2028,1210,8


### 6. Eight pairs

Exact covariates are copied onto both sides. They agree by construction. `distance` is the z-score Euclidean distance of the chosen neighbor.

In [6]:
show = ["distance"]
for column in spec["exact"]:
    show += [f"{column}_focal", f"{column}_comparison"]
for column in spec.get("caliper") or {}:
    gap = f"{column}_gap"
    pairs[gap] = (pairs[f"{column}_focal"] - pairs[f"{column}_comparison"]).abs()
    show.append(gap)
pairs[show].head(8)


,distance,gender_focal,gender_comparison,city_focal,city_comparison,category_focal,category_comparison,time_of_day_focal,time_of_day_comparison,upload_week_focal,upload_week_comparison,duration_seconds_gap
0,6.183805,female,female,north,north,food,food,noon,noon,week_2,week_2,1.845844
1,4.641880,female,female,north,north,food,food,noon,noon,week_2,week_2,2.216721
2,4.728320,female,female,north,north,food,food,noon,noon,week_2,week_2,2.777206
3,7.511530,female,female,north,north,food,food,noon,noon,week_2,week_2,0.827548
4,4.094523,female,female,north,north,food,food,noon,noon,week_2,week_2,2.700354
5,5.781511,female,female,north,north,food,food,noon,noon,week_3,week_3,2.684250
6,5.867640,female,female,north,north,food,food,noon,noon,week_3,week_3,0.384740
7,9.242077,female,female,north,north,food,food,noon,noon,week_3,week_3,1.566490


### 7. Balance before and after

Standardized mean difference of each covariate, on the raw arms and on the matched pairs.

In [7]:
from core.balance import balance_before_after

balance = balance_before_after(focal, comparison, pairs, list(spec["exact"]) + list(spec["numeric"]))
balance


,covariate,mean_focal_before,mean_comparison_before,standardized_difference_before,mean_focal_after,mean_comparison_after,standardized_difference_after
0,age,31.943388,32.037152,-0.012019,31.905186,32.050861,-0.019377
1,creator_followers,774.913916,776.607925,-0.002742,775.094217,730.507639,0.076896
2,creator_plays_14d,88.422992,88.125955,0.008368,88.586933,85.877726,0.079442
3,creator_plays_60d,264.844269,265.948113,-0.009887,265.498269,261.316058,0.038991
4,creator_uploads_14d,3.629452,3.532505,0.063588,3.615120,3.497372,0.079605
5,creator_uploads_60d,8.861475,8.923779,-0.016528,8.866177,8.633352,0.066321
6,creator_watch_time_14d,161.123644,160.866490,0.003933,162.030311,157.900206,0.065661
7,creator_watch_time_60d,486.042350,477.254091,0.044308,487.297661,463.137222,0.128073
8,duration_seconds,18.104832,18.125109,-0.003391,18.059137,18.071530,-0.002455
9,plays_labeled_14d,10.132037,10.120318,0.002163,10.078492,9.779537,0.059415


### 8. The contrast

Mean of focal minus comparison. The standard error is Abadie-Imbens, with up to five within-arm neighbors. The Hodges-Lehmann median and the Wilcoxon p-value are the rank summary beside that mean.

In [8]:
from core.imbens import contrast_table

effects = contrast_table(
    focal,
    comparison,
    pairs,
    spec["outcomes"],
    spec["exact"],
    spec["numeric"],
    n_neighbors=int(cfg.get("imbens_neighbors", 5)),
)
out = root / "data" / "results"
out.mkdir(parents=True, exist_ok=True)
pairs.to_csv(out / "consumer_pairs.csv", index=False)
effects.to_csv(out / "consumer_effects.csv", index=False)
balance.to_csv(out / "consumer_balance.csv", index=False)
effects


,outcome,n_pairs,hl_focal_minus_comparison,mean_difference,wilcoxon_statistic,wilcoxon_p,variance,standard_error,z_statistic,p_value,ci95_low,ci95_high
0,early_retention,2028,-0.029416,-0.029208,478484.0,1.176856e-96,0.000003,0.001654,-17.655664,0.0,-0.032450,-0.025966
1,valid_view_rate,2028,-0.057839,-0.058086,154650.0,7.628523e-241,0.000003,0.001707,-34.022055,0.0,-0.061433,-0.054740
2,full_view_rate,2028,-0.023088,-0.022978,449275.0,5.592157e-107,0.000001,0.001175,-19.555744,0.0,-0.025281,-0.020675
3,view_fraction,2028,-0.042747,-0.042540,393926.0,5.342872e-128,0.000004,0.002005,-21.216307,0.0,-0.046470,-0.038610
